# Kigurumi Head Reference Agent (v1)

End-to-end Agent that turns a target character + expression reference into:

1. **Step 1** — unobstructed head four-view (general prompt + ≤2 targeted refines).
2. **Dataset pick** — 0–2 visually similar finished shop products from `../dataset/`.
3. **Step 2** — kigurumi head shell four-view (general + ≤2 refines).
4. **Step 3** — product-photo four-view (single pass).

**Models**
- Image: `gpt-image-2` via OpenAI-compatible `images.edit`.
- Reasoning / vision: Qwen3.5-VL (`qwen-vl-max-latest`) via Aliyun Dashscope OpenAI-compatible endpoint.

**Setup**
```bash
cd notebook
pip install -r requirements.txt
cp .env.example .env   # then fill in IMAGE_API_KEY and REASON_API_KEY
```

Outputs land in `notebook/outputs/<timestamp>_<label>/` — every iteration's candidates plus a `run.json` audit trail.

In [ ]:
import importlib, prompts, config, agent
importlib.reload(prompts); importlib.reload(config); importlib.reload(agent)
from prompts import load_all
from config import load_config, DATASET_DIR, OUTPUT_DIR
from agent import KigurumiAgent

for step_id, sp in load_all().items():
    print(f'step{step_id}: general={len(sp.general)} chars, refines={sp.refinement_keys()}')

cfg = load_config()
print('Image:', cfg.image.model, '@', cfg.image.base_url)
print('Reason:', cfg.reason.model, '@', cfg.reason.base_url)
print('Dataset folders:', len(list(DATASET_DIR.iterdir())))

## Inputs

Fill in the three inputs below:
- `CHAR_REF_PATHS`: 1–4 official / structure references for the target character (front, back, side, etc.).
- `CHAR_INFO`: a short text block: IP, character name, key visual traits — used by the dataset picker and reviewer.
- `EXPR_REF_PATH`: one expression reference image.

Drop your reference images anywhere — `references/private/` (gitignored) is the recommended spot.


In [ ]:
CHAR_REF_PATHS = [
    '../references/private/char_front.png',
    '../references/private/char_back.png',
]
CHAR_INFO = '''原神 - 甘雨
半人半仙，浅蓝长直发带犄角，琥珀色虹膜，气质冷静温柔。
目标头壳：幼态圆脸，保留犄角和发尾装饰。'''
EXPR_REF_PATH = '../references/private/expression.png'
RUN_LABEL = 'ganyu-demo'

In [ ]:
agent_runner = KigurumiAgent(cfg)
result = agent_runner.run(
    char_ref_paths=CHAR_REF_PATHS,
    char_info=CHAR_INFO,
    expr_ref_path=EXPR_REF_PATH,
    run_label=RUN_LABEL,
)
print('Run dir:', result.run_dir)
print('Step 1 best:', result.step1.best.path)
print('Step 2 best:', result.step2.best.path)
print('Step 3 best:', result.step3.best.path)
print('Similar picks:', [p.name for p in result.similar])

## Inspect results inline

In [ ]:
from IPython.display import Image, display, Markdown

def show(label, path):
    display(Markdown(f'### {label}\n`{path}`'))
    display(Image(filename=str(path)))

show('Step 1 — unobstructed head turnaround', result.step1.best.path)
show('Step 2 — kigurumi shell turnaround', result.step2.best.path)
show('Step 3 — product-photo four-view', result.step3.best.path)

if result.similar:
    display(Markdown('### Similar shop products used as references'))
    for p in result.similar:
        show(p.stem, p)

## Audit trail

Every candidate and verdict is saved. `run.json` shows what was generated each iteration, what the reviewer said, and which refine prompt was picked.

In [ ]:
import json
audit = json.loads((result.run_dir / 'run.json').read_text(encoding='utf-8'))
for step in ('step1', 'step2', 'step3'):
    print(f'--- {step} ---')
    for it in audit[step]['iterations']:
        v = it.get('verdict', {})
        print(f"  {it['prompt_key']}: best={v.get('best_index')} verdict={v.get('verdict')} "
              f"refine={v.get('refine_key')} notes={v.get('notes')}")